# VacinaBR-PNI: ETL via CSVs mensais no Google Drive

Este notebook baixa os ZIPs mensais do Portal de Dados Abertos do SUS, salva em Google Drive e gera a base curada em Parquet, relatorios analiticos e documentacao.

Fonte: https://dadosabertos.saude.gov.br/dataset/doses-aplicadas-pelo-programa-de-nacional-de-imunizacoes-pni-2025

In [ ]:
# Célula 1 - Conecta o Google Colab ao Google Drive.
# Todos os arquivos grandes da pipeline ficam no Drive para persistirem
# depois que a sessão do Colab encerrar.
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Célula 2 - Instala a camada analítica local usada pela disciplina/projeto.
# DuckDB funciona como banco SQL embarcado sobre arquivos Parquet, sem servidor.
!pip install -q duckdb

In [ ]:
# Célula 3 - Importa bibliotecas e define a estrutura de pastas no Drive.
# raw/zip guarda os ZIPs originais, processed guarda Parquet curado,
# analytics guarda agregados CSV e docs guarda metadados/documentação.
from __future__ import annotations

import csv
import json
import os
import re
import shutil
import zipfile
from pathlib import Path
from typing import Any

import duckdb
import pandas as pd
import requests

DRIVE_ROOT = Path('/content/drive/MyDrive/VacinaBR-PNI')
RAW_ZIP_DIR = DRIVE_ROOT / 'data' / 'raw' / 'zip'
PROCESSED_DIR = DRIVE_ROOT / 'data' / 'processed'
ANALYTICS_DIR = DRIVE_ROOT / 'data' / 'analytics'
DOCS_DIR = DRIVE_ROOT / 'docs'
DUCKDB_PATH = DRIVE_ROOT / 'data' / 'vacinabr_pni.duckdb'

# CHUNKSIZE controla o uso de memória ao ler CSVs grandes.
# MAX_MONTHS=1 é seguro para teste; use None na execução final completa.
# DEDUPLICATE ativa remoção simples de duplicatas por hash de linha.
CHUNKSIZE = 100_000
MAX_MONTHS = 1  # Use 1 para teste. Troque para None para processar os 12 meses.
DEDUPLICATE = True
CSV_ENCODING = 'latin1'
RESET_GENERATED_OUTPUTS = True  # Limpa processed/analytics/docs/DuckDB, preservando raw/zip.

# Remove artefatos gerados em execuções anteriores para evitar ler Parquets antigos
# com schema incorreto. Os ZIPs brutos em raw/zip são preservados.
if RESET_GENERATED_OUTPUTS:
    for path in [PROCESSED_DIR, ANALYTICS_DIR, DOCS_DIR]:
        if path.exists():
            shutil.rmtree(path)
    if DUCKDB_PATH.exists():
        DUCKDB_PATH.unlink()

# Cria as pastas antes do download/processamento para evitar erros de caminho.
for path in [RAW_ZIP_DIR, PROCESSED_DIR, ANALYTICS_DIR, DOCS_DIR]:
    path.mkdir(parents=True, exist_ok=True)

print('Drive root:', DRIVE_ROOT)

In [ ]:
# Célula 4 - Define o manifesto das fontes oficiais.
# Cada item representa um ZIP mensal publicado pelo Portal de Dados Abertos do SUS.
# O manifesto também é salvo em docs/ para registrar exatamente quais arquivos
# alimentaram a versão processada do dataset.
MANIFEST = [
    {'year': 2025, 'month': 1, 'month_name': 'janeiro', 'url': 'https://s3.sa-east-1.amazonaws.com/ckan.saude.gov.br/PNI/csv/vacinacao_jan_2025_csv.zip'},
    {'year': 2025, 'month': 2, 'month_name': 'fevereiro', 'url': 'https://s3.sa-east-1.amazonaws.com/ckan.saude.gov.br/PNI/csv/vacinacao_fev_2025_csv.zip'},
    {'year': 2025, 'month': 3, 'month_name': 'marco', 'url': 'https://s3.sa-east-1.amazonaws.com/ckan.saude.gov.br/PNI/csv/vacinacao_mar_2025_csv.zip'},
    {'year': 2025, 'month': 4, 'month_name': 'abril', 'url': 'https://s3.sa-east-1.amazonaws.com/ckan.saude.gov.br/PNI/csv/vacinacao_abr_2025_csv.zip'},
    {'year': 2025, 'month': 5, 'month_name': 'maio', 'url': 'https://s3.sa-east-1.amazonaws.com/ckan.saude.gov.br/PNI/csv/vacinacao_mai_2025_csv.zip'},
    {'year': 2025, 'month': 6, 'month_name': 'junho', 'url': 'https://s3.sa-east-1.amazonaws.com/ckan.saude.gov.br/PNI/csv/vacinacao_jun_2025_csv.zip'},
    {'year': 2025, 'month': 7, 'month_name': 'julho', 'url': 'https://s3.sa-east-1.amazonaws.com/ckan.saude.gov.br/PNI/csv/vacinacao_jul_2025_csv.zip'},
    {'year': 2025, 'month': 8, 'month_name': 'agosto', 'url': 'https://s3.sa-east-1.amazonaws.com/ckan.saude.gov.br/PNI/csv/vacinacao_ago_2025_csv.zip'},
    {'year': 2025, 'month': 9, 'month_name': 'setembro', 'url': 'https://s3.sa-east-1.amazonaws.com/ckan.saude.gov.br/PNI/csv/vacinacao_set_2025_csv.zip'},
    {'year': 2025, 'month': 10, 'month_name': 'outubro', 'url': 'https://s3.sa-east-1.amazonaws.com/ckan.saude.gov.br/PNI/csv/vacinacao_out_2025_csv.zip'},
    {'year': 2025, 'month': 11, 'month_name': 'novembro', 'url': 'https://s3.sa-east-1.amazonaws.com/ckan.saude.gov.br/PNI/csv/vacinacao_nov_2025_csv.zip'},
    {'year': 2025, 'month': 12, 'month_name': 'dezembro', 'url': 'https://s3.sa-east-1.amazonaws.com/ckan.saude.gov.br/PNI/csv/vacinacao_dez_2025_csv.zip'},
]

# Transforma o manifesto em tabela para inspeção e reprodutibilidade.
manifest_df = pd.DataFrame(MANIFEST)
manifest_df['source_page'] = 'https://dadosabertos.saude.gov.br/dataset/doses-aplicadas-pelo-programa-de-nacional-de-imunizacoes-pni-2025'
manifest_df.to_csv(DOCS_DIR / 'pni_2025_csv_manifest.csv', index=False)
manifest_df

In [ ]:
# Célula 5 - Regras de padronização, limpeza, validação e enriquecimento.
# Esta célula não executa o ETL ainda; ela define funções reutilizadas na leitura
# em chunks para transformar cada pedaço do CSV bruto em dados curados.

# Mapeia UFs para regiões brasileiras, criando uma dimensão analítica derivada.
UF_TO_REGION = {
    'AC': 'Norte', 'AP': 'Norte', 'AM': 'Norte', 'PA': 'Norte', 'RO': 'Norte', 'RR': 'Norte', 'TO': 'Norte',
    'AL': 'Nordeste', 'BA': 'Nordeste', 'CE': 'Nordeste', 'MA': 'Nordeste', 'PB': 'Nordeste', 'PE': 'Nordeste', 'PI': 'Nordeste', 'RN': 'Nordeste', 'SE': 'Nordeste',
    'DF': 'Centro-Oeste', 'GO': 'Centro-Oeste', 'MT': 'Centro-Oeste', 'MS': 'Centro-Oeste',
    'ES': 'Sudeste', 'MG': 'Sudeste', 'RJ': 'Sudeste', 'SP': 'Sudeste',
    'PR': 'Sul', 'RS': 'Sul', 'SC': 'Sul',
}

# Remove identificadores sensíveis do dataset processado.
# Mantemos apenas campos necessários para análises agregadas epidemiológicas.
SENSITIVE_COLUMNS = ['codigo_paciente', 'numero_cep_paciente']

# Campos usados para medir completude mínima de um registro de vacinação.
ESSENTIAL_COLUMNS = ['data_vacina', 'sexo_paciente', 'numero_idade_paciente', 'uf_paciente', 'codigo_municipio_paciente', 'codigo_vacina', 'sg_vacina']

# Algumas colunas do SUS podem aparecer com nomes diferentes; os aliases
# convergem esses nomes para um schema único.
ALIASES = {
    'sigla_uf_paciente': 'uf_paciente',
    'sigla_uf_estabelecimento': 'uf_estabelecimento',
    'sigla_vacina': 'sg_vacina',
    'tipo_sexo_paciente': 'sexo_paciente',
    'status_documento': 'st_documento',
    'nome_razao_social_estabelecimento': 'razao_social_estabelecimento',
    'nome_fantasia_estalecimento': 'nome_fantasia_estabelecimento',
}

# O CSV mensal oficial do PNI 2025 vem sem cabeçalho. Esta lista define
# a ordem posicional dos 56 campos observados no arquivo real de janeiro.
CSV_COLUMNS = [
    'codigo_documento',
    'codigo_paciente',
    'tipo_sexo_paciente',
    'codigo_raca_cor_paciente',
    'nome_raca_cor_paciente',
    'codigo_municipio_paciente',
    'codigo_pais_paciente',
    'nome_municipio_paciente',
    'nome_pais_paciente',
    'sigla_uf_paciente',
    'numero_cep_paciente',
    'descricao_nacionalidade_paciente',
    'codigo_etnia_indigena_paciente',
    'nome_etnia_indigena_paciente',
    'codigo_cnes_estabelecimento',
    'nome_razao_social_estabelecimento',
    'nome_fantasia_estalecimento',
    'codigo_municipio_estabelecimento',
    'nome_municipio_estabelecimento',
    'sigla_uf_estabelecimento',
    'codigo_pais_estabelecimento',
    'codigo_vacina_fabricante',
    'sigla_vacina',
    'data_vacina',
    'numero_idade_paciente',
    'descricao_dose_vacina',
    'codigo_dose_vacina',
    'descricao_local_aplicacao',
    'codigo_via_administracao',
    'descricao_via_administracao',
    'codigo_lote_vacina',
    'descricao_vacina_fabricante',
    'data_entrada_rnds',
    'codigo_sistema_origem',
    'descricao_sistema_origem',
    'status_documento',
    'codigo_estrategia_vacinacao',
    'descricao_estrategia_vacinacao',
    'codigo_origem_registro',
    'descricao_origem_registro',
    'codigo_vacina_categoria_atendimento',
    'descricao_vacina_categoria_atendimento',
    'codigo_vacina_grupo_atendimento',
    'descricao_vacina_grupo_atendimento',
    'codigo_vacina',
    'descricao_vacina',
    'codigo_condicao_maternal',
    'codigo_tipo_estabelecimento',
    'descricao_tipo_estabelecimento',
    'codigo_natureza_estabelecimento',
    'descricao_natureza_estabelecimento',
    'codigo_troca_documento',
    'descricao_condicao_maternal',
    'nome_uf_paciente',
    'nome_uf_estabelecimento',
    'data_deletado_rnds',
]

def normalize_column_name(column: str) -> str:
    # Converte nomes de colunas para snake_case simples e previsível.
    column = str(column).strip().lower()
    column = re.sub(r'[^a-z0-9_]+', '_', column)
    column = re.sub(r'_+', '_', column).strip('_')
    return column

def normalize_text(series: pd.Series) -> pd.Series:
    # Padroniza textos categóricos: remove espaços, usa maiúsculas e troca vazios por NA.
    return series.astype('string').str.strip().str.upper().replace({'': pd.NA, 'NAN': pd.NA, 'NONE': pd.NA, 'NULL': pd.NA})

def age_group(age: Any) -> str | None:
    # Cria faixas etárias consistentes para análises agregadas.
    if pd.isna(age):
        return None
    try:
        age_int = int(age)
    except (TypeError, ValueError):
        return None
    if age_int < 0 or age_int > 130:
        return 'idade_invalida'
    if age_int <= 1: return '0-1'
    if age_int <= 4: return '2-4'
    if age_int <= 11: return '5-11'
    if age_int <= 17: return '12-17'
    if age_int <= 29: return '18-29'
    if age_int <= 39: return '30-39'
    if age_int <= 59: return '40-59'
    return '60+'

def transform_chunk(df: pd.DataFrame) -> pd.DataFrame:
    # Copia o chunk para evitar efeitos colaterais no leitor incremental.
    df = df.copy()

    # Padroniza nomes e aplica aliases antes de validar campos.
    df.columns = [normalize_column_name(c) for c in df.columns]
    df = df.rename(columns={src: dst for src, dst in ALIASES.items() if src in df.columns and dst not in df.columns})

    # Converte data e idade para tipos analíticos; valores inválidos viram NA.
    if 'data_vacina' in df.columns:
        df['data_vacina'] = pd.to_datetime(df['data_vacina'], errors='coerce')
    if 'numero_idade_paciente' in df.columns:
        df['numero_idade_paciente'] = pd.to_numeric(df['numero_idade_paciente'], errors='coerce')

    # Normaliza campos categóricos para reduzir variações de grafia/capitalização.
    text_columns = ['sexo_paciente', 'uf_paciente', 'uf_estabelecimento', 'nome_uf_paciente', 'nome_uf_estabelecimento', 'nome_municipio_paciente', 'sg_vacina', 'descricao_dose_vacina', 'descricao_vacina_fabricante', 'descricao_estrategia_vacinacao', 'descricao_sistema_origem', 'st_documento']
    for col in text_columns:
        if col in df.columns:
            df[col] = normalize_text(df[col])

    # Preserva códigos como texto para não perder zeros à esquerda.
    for col in ['codigo_municipio_paciente', 'codigo_municipio_estabelecimento', 'codigo_vacina', 'codigo_sistema_origem']:
        if col in df.columns:
            df[col] = df[col].astype('string').str.strip().replace({'': pd.NA})

    # Cria flags de qualidade documental: registro final e não deletado na RNDS.
    df['registro_deletado_rnds'] = df['data_deletado_rnds'].notna() if 'data_deletado_rnds' in df.columns else False
    df['documento_final'] = df['st_documento'].fillna('').str.upper().eq('FINAL') if 'st_documento' in df.columns else True
    df['registro_valido_documento'] = df['documento_final'] & (~df['registro_deletado_rnds'])

    # Deriva dimensões temporais para agregações por ano, mês, trimestre e semana.
    if 'data_vacina' in df.columns:
        df['ano_vacinacao'] = df['data_vacina'].dt.year.astype('Int64')
        df['mes_vacinacao'] = df['data_vacina'].dt.month.astype('Int64')
        df['trimestre_vacinacao'] = 'Q' + df['data_vacina'].dt.quarter.astype('Int64').astype('string')
        df['semana_epidemiologica'] = df['data_vacina'].dt.isocalendar().week.astype('Int64')
    else:
        df['ano_vacinacao'] = pd.NA
        df['mes_vacinacao'] = pd.NA
        df['trimestre_vacinacao'] = pd.NA
        df['semana_epidemiologica'] = pd.NA

    # Valida idade e cria faixa etária derivada.
    if 'numero_idade_paciente' in df.columns:
        df['idade_valida'] = df['numero_idade_paciente'].between(0, 130, inclusive='both')
        df['faixa_etaria'] = df['numero_idade_paciente'].apply(age_group)
    else:
        df['idade_valida'] = False
        df['faixa_etaria'] = pd.NA

    # Enriquece os registros com região geográfica do paciente e do estabelecimento.
    df['regiao_paciente'] = df['uf_paciente'].map(UF_TO_REGION).fillna('Nao informado') if 'uf_paciente' in df.columns else 'Nao informado'
    df['regiao_estabelecimento'] = df['uf_estabelecimento'].map(UF_TO_REGION).fillna('Nao informado') if 'uf_estabelecimento' in df.columns else 'Nao informado'

    # Flag útil para estudar deslocamento entre município do paciente e local de aplicação.
    if {'codigo_municipio_paciente', 'codigo_municipio_estabelecimento'}.issubset(df.columns):
        df['municipio_paciente_igual_estabelecimento'] = df['codigo_municipio_paciente'] == df['codigo_municipio_estabelecimento']
    else:
        df['municipio_paciente_igual_estabelecimento'] = pd.NA

    # Calcula completude e remove colunas sensíveis antes de salvar Parquet.
    existing_essential = [c for c in ESSENTIAL_COLUMNS if c in df.columns]
    df['registro_completo'] = df[existing_essential].notna().all(axis=1) if existing_essential else False
    df = df.drop(columns=[c for c in SENSITIVE_COLUMNS if c in df.columns], errors='ignore')
    return df

In [ ]:
# Célula 6 - Baixa os ZIPs mensais e identifica o CSV dentro de cada ZIP.
# Os downloads são salvos em raw/zip no Drive e reutilizados se já existirem,
# evitando baixar novamente arquivos grandes em execuções posteriores.

def download_file(url: str, output_path: Path) -> None:
    # Reaproveita o arquivo se ele já foi baixado com tamanho maior que zero.
    if output_path.exists() and output_path.stat().st_size > 0:
        print('[download] reuse', output_path.name)
        return
    print('[download]', url)

    # Grava primeiro em .part para não deixar um ZIP incompleto com nome final.
    tmp_path = output_path.with_suffix(output_path.suffix + '.part')
    with requests.get(url, stream=True, timeout=120) as response:
        response.raise_for_status()
        with tmp_path.open('wb') as file:
            for chunk in response.iter_content(chunk_size=1024 * 1024):
                if chunk:
                    file.write(chunk)
    tmp_path.replace(output_path)

def zip_csv_member(zip_path: Path) -> str:
    # Localiza o primeiro arquivo CSV dentro do ZIP mensal.
    with zipfile.ZipFile(zip_path) as zf:
        csv_names = [name for name in zf.namelist() if name.lower().endswith('.csv')]
        if not csv_names:
            raise RuntimeError(f'Nenhum CSV encontrado em {zip_path}')
        return csv_names[0]

def detect_sep(zip_path: Path, member: str) -> str:
    # Detecta delimitador por contagem simples no cabeçalho: ';' ou ','.
    with zipfile.ZipFile(zip_path) as zf:
        with zf.open(member) as file:
            header = file.readline().decode(CSV_ENCODING, errors='replace')
    return ';' if header.count(';') >= header.count(',') else ','

def csv_has_header(zip_path: Path, member: str, sep: str) -> bool:
    # Janeiro/2025 foi publicado sem cabeçalho. Esta função diferencia
    # arquivos com header de arquivos puramente posicionais.
    with zipfile.ZipFile(zip_path) as zf:
        with zf.open(member) as file:
            text = file.readline().decode(CSV_ENCODING, errors='replace')
    first_row = next(csv.reader([text], delimiter=sep))
    normalized = {normalize_column_name(value) for value in first_row}
    return bool({'data_vacina', 'codigo_documento', 'tipo_sexo_paciente'} & normalized)

# Seleciona só os primeiros meses durante teste ou todos quando MAX_MONTHS=None.
selected_manifest = MANIFEST[:MAX_MONTHS] if MAX_MONTHS else MANIFEST
for item in selected_manifest:
    filename = Path(item['url']).name
    download_file(item['url'], RAW_ZIP_DIR / filename)

In [ ]:
# Célula 7 - Executa o ETL principal em chunks e salva Parquet particionado.
# Esta etapa lê os CSVs grandes sem carregar tudo em memória, aplica as regras
# da célula anterior, calcula métricas de qualidade e persiste os dados curados.

# Métricas globais que alimentarão o relatório de validação do dataset.
metrics = {
    'original_records': 0,
    'processed_records': 0,
    'removed_duplicate_records': 0,
    'total_missing_values': 0,
    'invalid_age_records': 0,
    'invalid_date_records': 0,
    'complete_records': 0,
    'incomplete_records': 0,
    'valid_document_records': 0,
    'invalid_document_records': 0,
}

# missing_metrics guarda ausências por coluna-chave; seen_hashes apoia deduplicação.
# schema_columns captura o schema observado no primeiro chunk processado.
missing_metrics = {}
seen_hashes = set()
schema_columns = None

def update_metrics(df: pd.DataFrame, original_rows: int, removed_duplicates: int) -> None:
    # Atualiza contadores de volume, qualidade, completude e validade documental.
    metrics['original_records'] += original_rows
    metrics['processed_records'] += len(df)
    metrics['removed_duplicate_records'] += removed_duplicates
    metrics['total_missing_values'] += int(df.isna().sum().sum())
    if 'idade_valida' in df.columns:
        metrics['invalid_age_records'] += int((~df['idade_valida'].fillna(False)).sum())
    if 'data_vacina' in df.columns:
        metrics['invalid_date_records'] += int(df['data_vacina'].isna().sum())
    if 'registro_completo' in df.columns:
        metrics['complete_records'] += int(df['registro_completo'].sum())
        metrics['incomplete_records'] += int((~df['registro_completo'].fillna(False)).sum())
    if 'registro_valido_documento' in df.columns:
        metrics['valid_document_records'] += int(df['registro_valido_documento'].sum())
        metrics['invalid_document_records'] += int((~df['registro_valido_documento'].fillna(False)).sum())
    for col in ['data_vacina', 'sexo_paciente', 'numero_idade_paciente', 'uf_paciente', 'codigo_municipio_paciente', 'sg_vacina', 'descricao_vacina_fabricante']:
        if col in df.columns:
            missing_metrics[f'missing_{col}'] = missing_metrics.get(f'missing_{col}', 0) + int(df[col].isna().sum())

def deduplicate_chunk(df: pd.DataFrame) -> tuple[pd.DataFrame, int]:
    # Remove duplicatas entre chunks usando hash da linha sem campos sensíveis.
    if not DEDUPLICATE or df.empty:
        return df, 0
    dedup_subset = [c for c in df.columns if c not in SENSITIVE_COLUMNS]
    hashes = pd.util.hash_pandas_object(df[dedup_subset].astype('string'), index=False).astype('uint64')
    duplicate_mask = hashes.map(lambda value: int(value) in seen_hashes)
    new_hashes = hashes[~duplicate_mask]
    seen_hashes.update(int(value) for value in new_hashes)
    return df.loc[~duplicate_mask].copy(), int(duplicate_mask.sum())

part_counter = 0
for item in selected_manifest:
    # Para cada mês, abre o ZIP, detecta o separador e inicia leitura incremental.
    zip_path = RAW_ZIP_DIR / Path(item['url']).name
    member = zip_csv_member(zip_path)
    sep = detect_sep(zip_path, member)
    has_header = csv_has_header(zip_path, member, sep)
    header = 0 if has_header else None
    names = None if has_header else CSV_COLUMNS
    print(f'[process] {zip_path.name} member={member} sep={sep!r} header={has_header}')
    with zipfile.ZipFile(zip_path) as zf:
        with zf.open(member) as file:
            reader = pd.read_csv(file, sep=sep, header=header, names=names, dtype='string', chunksize=CHUNKSIZE, encoding=CSV_ENCODING, low_memory=False)
            for chunk_index, chunk in enumerate(reader):
                # Cada chunk passa por transformação, deduplicação e validação.
                original_rows = len(chunk)
                processed = transform_chunk(chunk)
                processed, removed_duplicates = deduplicate_chunk(processed)
                update_metrics(processed, original_rows, removed_duplicates)
                if schema_columns is None:
                    # Registra o schema curado para documentação posterior.
                    schema_columns = [{'name': c, 'pandas_dtype': str(t)} for c, t in processed.dtypes.items()]
                if processed.empty:
                    continue
                year = int(processed['ano_vacinacao'].dropna().iloc[0]) if processed['ano_vacinacao'].notna().any() else item['year']
                month = int(processed['mes_vacinacao'].dropna().iloc[0]) if processed['mes_vacinacao'].notna().any() else item['month']

                # Particiona por ano/mês para acelerar consultas e reduzir leitura futura.
                out_dir = PROCESSED_DIR / f'year={year}' / f'month={month:02d}'
                out_dir.mkdir(parents=True, exist_ok=True)
                out_path = out_dir / f'part-{part_counter:06d}.parquet'
                processed.to_parquet(out_path, index=False)
                part_counter += 1
                if chunk_index % 10 == 0:
                    print(f'  chunk={chunk_index} rows={len(processed):,} saved={out_path.name}')

print('[done] parquet parts:', part_counter)

In [ ]:
# Célula 8 - Gera artefatos finais: agregados, documentação e banco DuckDB.
# A saída desta célula é o pacote consumível do projeto: CSVs analíticos,
# relatórios de qualidade, dicionário de dados, metadados e arquivo .duckdb.

def load_processed_columns(columns: list[str] | None = None) -> pd.DataFrame:
    # Carrega apenas as colunas necessárias dos Parquets para economizar memória.
    files = sorted(PROCESSED_DIR.glob('year=*/month=*/*.parquet'))
    frames = [pd.read_parquet(path, columns=columns) for path in files]
    return pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()

def save_summary(group_cols: list[str], filename: str) -> None:
    # Cria uma tabela agregada por dimensões específicas e salva como CSV.
    cols = list(dict.fromkeys(group_cols))
    df = load_processed_columns(cols)
    existing = [c for c in group_cols if c in df.columns]
    if not existing or df.empty:
        return
    result = df.groupby(existing, dropna=False).size().reset_index(name='doses_aplicadas')
    result.to_csv(ANALYTICS_DIR / filename, index=False, encoding='utf-8')

# Agregados de análise epidemiológica: tempo, UF, região, vacina, fabricante,
# faixa etária, sexo e cruzamentos úteis para exploração inicial.
save_summary(['ano_vacinacao', 'mes_vacinacao'], 'monthly_vaccination_summary.csv')
save_summary(['uf_paciente'], 'state_vaccination_summary.csv')
save_summary(['regiao_paciente'], 'region_vaccination_summary.csv')
save_summary(['sg_vacina'], 'vaccine_type_summary.csv')
save_summary(['descricao_vacina_fabricante'], 'manufacturer_summary.csv')
save_summary(['faixa_etaria'], 'age_group_summary.csv')
save_summary(['sexo_paciente'], 'sex_summary.csv')
save_summary(['ano_vacinacao', 'mes_vacinacao', 'sg_vacina'], 'monthly_vaccine_type_summary.csv')
save_summary(['uf_paciente', 'sg_vacina'], 'state_vaccine_type_summary.csv')

# Relatório de validação: consolida métricas de volume, duplicatas,
# ausências, idades inválidas, datas inválidas e registros documentais válidos.
validation_rows = [
    {'metric': key, 'value': value, 'description': key.replace('_', ' ')}
    for key, value in {**metrics, **missing_metrics}.items()
]
pd.DataFrame(validation_rows).to_csv(DOCS_DIR / 'validation_report.csv', index=False, encoding='utf-8')

# Schema observado na base curada, útil para auditoria e para o data paper.
schema = {
    'dataset': 'VacinaBR-PNI',
    'source': 'Portal de Dados Abertos do SUS - CSVs mensais',
    'columns': schema_columns or [],
}
with (DOCS_DIR / 'schema.json').open('w', encoding='utf-8') as file:
    json.dump(schema, file, ensure_ascii=False, indent=2)

# Dicionário de dados dos principais campos brutos preservados e campos derivados.
data_dictionary = [
    ('data_vacina', 'date', 'Data de aplicacao da vacina.'),
    ('ano_vacinacao', 'integer', 'Ano extraido da data de vacinacao.'),
    ('mes_vacinacao', 'integer', 'Mes extraido da data de vacinacao.'),
    ('trimestre_vacinacao', 'string', 'Trimestre da vacinacao.'),
    ('semana_epidemiologica', 'integer', 'Semana epidemiologica estimada a partir da data de vacinacao.'),
    ('sexo_paciente', 'string', 'Sexo informado do paciente.'),
    ('numero_idade_paciente', 'number', 'Idade informada do paciente.'),
    ('faixa_etaria', 'string', 'Faixa etaria derivada da idade.'),
    ('idade_valida', 'boolean', 'Indica se a idade esta no intervalo esperado de 0 a 130 anos.'),
    ('uf_paciente', 'string', 'Unidade federativa do paciente.'),
    ('regiao_paciente', 'string', 'Regiao geografica derivada da UF do paciente.'),
    ('codigo_municipio_paciente', 'string', 'Codigo do municipio do paciente.'),
    ('uf_estabelecimento', 'string', 'Unidade federativa do estabelecimento.'),
    ('regiao_estabelecimento', 'string', 'Regiao geografica derivada da UF do estabelecimento.'),
    ('codigo_municipio_estabelecimento', 'string', 'Codigo do municipio do estabelecimento.'),
    ('municipio_paciente_igual_estabelecimento', 'boolean', 'Indica se paciente e estabelecimento pertencem ao mesmo municipio.'),
    ('codigo_vacina', 'string', 'Codigo da vacina.'),
    ('sg_vacina', 'string', 'Sigla da vacina ou imunobiologico.'),
    ('descricao_dose_vacina', 'string', 'Descricao da dose aplicada.'),
    ('descricao_vacina_fabricante', 'string', 'Fabricante informado da vacina.'),
    ('descricao_estrategia_vacinacao', 'string', 'Estrategia de vacinacao informada.'),
    ('descricao_sistema_origem', 'string', 'Sistema de origem do registro.'),
    ('st_documento', 'string', 'Status documental do registro.'),
    ('registro_deletado_rnds', 'boolean', 'Indica se o registro possui data de delecao na RNDS.'),
    ('documento_final', 'boolean', 'Indica se o status do documento e final.'),
    ('registro_valido_documento', 'boolean', 'Indica documento final e nao deletado.'),
    ('registro_completo', 'boolean', 'Indica preenchimento dos campos essenciais.'),
]
pd.DataFrame(data_dictionary, columns=['field', 'type', 'description']).to_csv(DOCS_DIR / 'data_dictionary.csv', index=False, encoding='utf-8')

# Metadados de proveniência: fonte oficial, recursos usados, locais de armazenamento
# e descrição da camada DuckDB usada para consultas SQL.
source_metadata = {
    'dataset': 'VacinaBR-PNI',
    'source_page': manifest_df['source_page'].iloc[0],
    'resources': MANIFEST,
    'raw_storage': str(RAW_ZIP_DIR),
    'processed_storage': str(PROCESSED_DIR),
    'database_layer': {
        'database': 'DuckDB',
        'database_path': str(DUCKDB_PATH),
        'purpose': 'Consultar os Parquets curados e materializar agregados analiticos por SQL.',
    },
    'sensitive_fields_removed_from_processed': SENSITIVE_COLUMNS,
}
with (DOCS_DIR / 'source_metadata.json').open('w', encoding='utf-8') as file:
    json.dump(source_metadata, file, ensure_ascii=False, indent=2)

# DuckDB: cria uma view SQL sobre todos os Parquets particionados.
# A view evita duplicar os dados curados e permite consultar o dataset por SQL.
parquet_pattern = str(PROCESSED_DIR / 'year=*' / 'month=*' / '*.parquet')
con = duckdb.connect(str(DUCKDB_PATH))
con.execute(f"""
CREATE OR REPLACE VIEW vacinacao_curada AS
SELECT *
FROM read_parquet('{parquet_pattern}', union_by_name = true)
""")
con.execute("""
CREATE OR REPLACE TABLE dataset_metadata AS
SELECT
    'VacinaBR-PNI' AS dataset,
    'Portal de Dados Abertos do SUS - CSVs mensais' AS source,
    current_timestamp AS generated_at,
    (SELECT count(*) FROM vacinacao_curada) AS total_records
""")

# Consultas SQL que materializam tabelas de resumo dentro do arquivo .duckdb.
duckdb_summary_queries = {
    'monthly_vaccination_summary': """
        SELECT ano_vacinacao, mes_vacinacao, count(*) AS doses_aplicadas
        FROM vacinacao_curada
        GROUP BY ano_vacinacao, mes_vacinacao
        ORDER BY ano_vacinacao, mes_vacinacao
    """,
    'state_vaccination_summary': """
        SELECT uf_paciente, count(*) AS doses_aplicadas
        FROM vacinacao_curada
        GROUP BY uf_paciente
        ORDER BY doses_aplicadas DESC
    """,
    'region_vaccination_summary': """
        SELECT regiao_paciente, count(*) AS doses_aplicadas
        FROM vacinacao_curada
        GROUP BY regiao_paciente
        ORDER BY doses_aplicadas DESC
    """,
    'vaccine_type_summary': """
        SELECT sg_vacina, count(*) AS doses_aplicadas
        FROM vacinacao_curada
        GROUP BY sg_vacina
        ORDER BY doses_aplicadas DESC
    """,
    'manufacturer_summary': """
        SELECT descricao_vacina_fabricante, count(*) AS doses_aplicadas
        FROM vacinacao_curada
        GROUP BY descricao_vacina_fabricante
        ORDER BY doses_aplicadas DESC
    """,
    'age_group_summary': """
        SELECT faixa_etaria, count(*) AS doses_aplicadas
        FROM vacinacao_curada
        GROUP BY faixa_etaria
        ORDER BY doses_aplicadas DESC
    """,
    'sex_summary': """
        SELECT sexo_paciente, count(*) AS doses_aplicadas
        FROM vacinacao_curada
        GROUP BY sexo_paciente
        ORDER BY doses_aplicadas DESC
    """,
}
for table_name, query in duckdb_summary_queries.items():
    # Cada tabela é recriada para refletir a versão mais recente dos Parquets.
    con.execute(f'CREATE OR REPLACE TABLE {table_name} AS {query}')

# Catálogo simples para documentar quais objetos existem no DuckDB final.
duckdb_catalog = {
    'database': str(DUCKDB_PATH),
    'source_parquet_pattern': parquet_pattern,
    'view': 'vacinacao_curada',
    'tables': [
        row[0]
        for row in con.execute("""
            SELECT table_name
            FROM information_schema.tables
            WHERE table_schema = 'main'
            ORDER BY table_name
        """).fetchall()
    ],
}
with (DOCS_DIR / 'duckdb_catalog.json').open('w', encoding='utf-8') as file:
    json.dump(duckdb_catalog, file, ensure_ascii=False, indent=2)

# Prévia final para conferir rapidamente se o banco e os agregados foram gerados.
print('[done] DuckDB:', DUCKDB_PATH)
display(con.sql('SELECT * FROM dataset_metadata').df())
display(con.sql('SELECT * FROM state_vaccination_summary LIMIT 10').df())
con.close()

print('[done] Analytics:', ANALYTICS_DIR)
print('[done] Docs:', DOCS_DIR)
pd.read_csv(DOCS_DIR / 'validation_report.csv').head(20)